# 12. Q21 как таргет и асимметричный штраф

Разбор GPU-эксперимента: прогноз Q21 на 1/3/6 ч, риск превышения 10 ppm, чувствительность к коду 307 и цена штрафа за пропуск превышения.

In [ ]:
from pathlib import Path
import pandas as pd
import plotly.express as px
from IPython.display import display
HERE=Path.cwd().resolve()
EDA=HERE if HERE.name=='eda' else HERE/'eda'
OUT=EDA/'experiments'/'q21_target_asymmetric_v3_20260915'
reg=pd.read_csv(OUT/'regression_metrics.csv')
cls=pd.read_csv(OUT/'classification_metrics.csv')
sens=pd.read_csv(OUT/'q21_307_sensitivity.csv')
thresholds=pd.read_csv(OUT/'asymmetric_threshold_metrics.csv')
baseline=pd.read_csv(OUT/'persistence_baseline.csv')

## Модели, выбранные только по validation

In [ ]:
selected=pd.read_csv(OUT/'selected_on_validation.csv')
display(selected[['horizon_h','model','feature_set','seed','ap','roc_auc','brier']])
display(baseline)

## Проверка без Q21=307

In [ ]:
clean=sens[(sens['split']=='evaluation')&(sens['population']=='exclude_307')]
display(clean[['horizon_h','feature_set','n','prevalence','ap','roc_auc','brier']])
fig=px.bar(clean,x='horizon_h',y='ap',color='feature_set',text_auto='.3f',title='AP риска Q21 > 10 ppm без кода 307')
fig.update_xaxes(title='Горизонт, ч'); fig.update_yaxes(title='Average precision'); fig.show()

## Компромисс безопасности и ложных тревог

In [ ]:
trade=sens[(sens['split']=='evaluation_threshold')].copy()
trade['penalty']=trade['population'].str.extract(r'(\d+)$')[0].astype(int)
display(trade[trade.horizon_h==1][['penalty','threshold','recall','precision','fpr','fn','fp','cost']])
long=trade.melt(id_vars=['horizon_h','penalty'],value_vars=['recall','precision','fpr'],var_name='metric',value_name='value')
fig=px.line(long,x='penalty',y='value',color='metric',facet_col='horizon_h',markers=True,title='Цена увеличения штрафа за пропуск превышения')
fig.update_yaxes(tickformat='.0%'); fig.show()

## Важность признаков выбранных моделей

Высокая важность `Q21_invalid_or_offscale` на 3–6 ч показывает, что код 307 необходимо анализировать отдельно от обычного содержания серы.

In [ ]:
importance=pd.read_csv(OUT/'selected_feature_importance.csv')
top=importance.groupby('horizon_h',group_keys=False).head(12)
fig=px.bar(top,x='importance',y='feature',facet_col='horizon_h',color='horizon_h',orientation='h',title='Основные признаки выбранных моделей')
fig.update_yaxes(matches=None,showticklabels=True); fig.show()